Landing / bronze checks – run by hand, not part of the ADF job chain. One section per source; each section defines what it needs. Moved out of 00-04.

In [0]:
import json, re, time, itertools
from collections import Counter
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from zoneinfo import ZoneInfo
import requests
import pandas as pd

# Shared helper – keep identical in 01–05
def http_call(api, call, method, url, ok=(200,), tries=4, timeout=30, **kw):
    """Retry on 429 / 5xx / timeouts / connection errors with backoff (2, 4, 8 s), then raise.
       Other 4xx = our error: no retry. Errors name the API, the call and the status.
       Test hook: a global SIMULATE_HTTP_STATUS (digits) makes every response that status, no request sent."""
    last = None
    for attempt in range(1, tries + 1):
        sim = str(globals().get("SIMULATE_HTTP_STATUS", "")).strip()
        try:
            if sim.isdigit():
                r = requests.Response(); r.status_code = int(sim); r._content = b""
            else:
                r = requests.request(method, url, timeout=timeout, **kw)
        except (requests.Timeout, requests.ConnectionError) as e:
            last = f"{type(e).__name__}: {e}"
        else:
            tag = " (simulated)" if sim.isdigit() else ""
            if r.status_code in ok:
                return r
            last = f"HTTP {r.status_code}{tag} {r.text[:300]}"
            if r.status_code != 429 and r.status_code < 500:
                break
        if attempt < tries:
            print(f"{api} | {call} | attempt {attempt}/{tries} failed: {last} – retrying in {2 ** attempt}s")
            time.sleep(2 ** attempt)
    raise RuntimeError(f"{api} | {call} | failed after {attempt} attempt(s): {last}")

00 – elprisetjustnu API test (from 00_Setup c1)

In [0]:
BASE = "https://www.elprisetjustnu.se/api/v1/prices"

def price_url(d: date, zone: str) -> str:
    return f"{BASE}/{d:%Y}/{d:%m-%d}_{zone}.json"

# (date, what we expect to see)
tests = [
    (date(2025, 9, 30),  "last hourly day   -> 24 rows"),
    (date(2025, 10, 1),  "first 15-min day  -> 96 rows"),
    (date(2025, 10, 26), "DST ends (25 h)   -> 100 rows"),
    (date(2026, 9, 23),  "normal recent day -> 96 rows"),
]

for d, expect in tests:
    r = requests.get(price_url(d, "SE3"), timeout=30)
    rows = r.json() if r.status_code == 200 else []
    print(f"{d}  HTTP {r.status_code}  rows={len(rows):>3}  expected: {expect}")

# Field names and one example row
print(json.dumps(rows[:1], indent=2))

01 elpris – loader (copy of 01_landing_elpris; keep in sync), test run, data check, bronze check

In [0]:
EL_API   = "elprisetjustnu"
EL_BASE  = "https://www.elprisetjustnu.se/api/v1/prices"
EL_ROOT  = Path("/Volumes/laddstolpar_df/landing/raw/elpris")
ZONES    = ["SE1", "SE2", "SE3", "SE4"]
START    = date(2025, 10, 1)      # first quarter-hour day (decision 2026-09-23)
EL_PAUSE = 0.2
TZ       = ZoneInfo("Europe/Stockholm")

def last_available_day(now=None):
    """Tomorrow's prices are published at 13:00 at the earliest."""
    now = now or datetime.now(TZ)
    return now.date() + timedelta(days=1) if now.hour >= 13 else now.date()

def target_path(d: date, zone: str) -> Path:
    return EL_ROOT / zone / f"{d:%Y}" / f"{d:%Y-%m-%d}_{zone}.json"

def land_elpris(start=START, end=None, zones=ZONES, max_calls=None):
    end = end or last_available_day()
    stats = {"calls": 0, "written": 0, "skipped": 0, "not_published": 0, "missing_past": []}
    d = start
    while d <= end:
        for z in zones:
            p = target_path(d, z)
            if p.exists() and p.stat().st_size > 0:
                stats["skipped"] += 1
                continue
            if max_calls is not None and stats["calls"] >= max_calls:
                return stats
            url = f"{EL_BASE}/{d:%Y}/{d:%m-%d}_{z}.json"
            r = http_call(EL_API, f"GET {url}", "GET", url, ok=(200, 404))
            stats["calls"] += 1
            if r.status_code == 404:
                stats["not_published"] += 1
                if d < end:
                    stats["missing_past"].append(f"{d}_{z}")
            else:
                r.json()
                p.parent.mkdir(parents=True, exist_ok=True)
                p.write_text(r.text, encoding="utf-8")
                stats["written"] += 1
            time.sleep(EL_PAUSE)
        d += timedelta(days=1)
    return stats

In [0]:
# Small test run (on a fresh volume: run 1 = calls=12 written=12, run 2 = calls=0 skipped=12)
test = dict(start=date(2026, 9, 20), end=date(2026, 9, 22))

print("Run 1:", land_elpris(**test))
print("Run 2:", land_elpris(**test))

for p in sorted(EL_ROOT.rglob("*.json")):
    print(p, p.stat().st_size, "bytes")

In [0]:
# Data check: every day x zone landed, right number of quarter-hours per local day
def quarter_hours_in_local_day(d: date) -> int:
    start = datetime(d.year, d.month, d.day, tzinfo=TZ).astimezone(timezone.utc)
    nxt = d + timedelta(days=1)
    end = datetime(nxt.year, nxt.month, nxt.day, tzinfo=TZ).astimezone(timezone.utc)
    return int((end - start).total_seconds() // 900)

end = last_available_day()
expected_days = [START + timedelta(days=i) for i in range((end - START).days + 1)]

problems, row_counts = [], Counter()
for z in ZONES:
    present = {p.name for p in (EL_ROOT / z).rglob("*.json")}
    wanted  = {target_path(d, z).name for d in expected_days}
    missing, extra = sorted(wanted - present), sorted(present - wanted)
    if missing: problems.append(f"{z} missing {len(missing)}: {missing[:5]}")
    if extra:   problems.append(f"{z} unexpected {len(extra)}: {extra[:5]}")
    for d in expected_days:
        p = target_path(d, z)
        if p.exists():
            n = len(json.loads(p.read_text()))
            row_counts[n] += 1
            if n != quarter_hours_in_local_day(d):
                problems.append(f"{p.name}: {n} rows, expected {quarter_hours_in_local_day(d)}")

print(f"Days: {len(expected_days)}  ×  zones: {len(ZONES)}  =  {len(expected_days)*len(ZONES)} files expected")
print("Rows per file (rows: files):", dict(sorted(row_counts.items())))
print("Problems:", problems or "none")

In [0]:
%sql
SELECT COUNT(*)                            AS rows_,
       COUNT(DISTINCT _file)               AS files,
       COUNT_IF(elomrade = '')             AS rows_without_zone,
       COUNT_IF(_rescued_data IS NOT NULL) AS rescued,
       COUNT_IF(SEK_per_kWh < 0)           AS negative_prices
FROM laddstolpar_df.bronze.elpris;

In [0]:
%sql DESCRIBE HISTORY laddstolpar_df.bronze.elpris

02 SCB – metadata overview, bronze checks, extra checks

In [0]:
SCB_NAME = "SCB"
SCB_API  = "https://statistikdatabasen.scb.se/api/v2"
SCB_TABLES = ["TAB3277", "TAB628", "TAB3276"]

def scb_get_json(path):
    return http_call(SCB_NAME, f"GET {path}", "GET", f"{SCB_API}/{path}", params={"lang": "sv"}).json()

def codes(meta, var_id):
    idx = meta["dimension"][var_id]["category"]["index"]
    return list(idx.keys()) if isinstance(idx, dict) else list(idx)

print("maxDataCells:", http_call(SCB_NAME, "GET config", "GET", f"{SCB_API}/config").json()["maxDataCells"])

for tab in SCB_TABLES:
    info = scb_get_json(f"tables/{tab}")
    meta = scb_get_json(f"tables/{tab}/metadata")
    print(f"\n{tab}  updated={info.get('updated')}  periods={info.get('firstPeriod')}–{info.get('lastPeriod')}")
    total = 1
    for v in meta["id"]:
        c = codes(meta, v)
        total *= len(c)
        print(f"  {v:<15} {len(c):>4} values   e.g. {c[:4]} … {c[-2:]}")
    print(f"  full table = {total:,} cells")

m3277 = scb_get_json("tables/TAB3277/metadata")
m628  = scb_get_json("tables/TAB628/metadata")
m3276 = scb_get_json("tables/TAB3276/metadata")

extra = sorted(set(codes(m3277, "Region")) - set(codes(m628, "Region")))
labels = m3277["dimension"]["Region"]["category"]["label"]
print("\nRegion codes in TAB3277 but not TAB628:", [(c, labels[c]) for c in extra])

In [0]:
%sql
SELECT 'scb_tab628' AS t, COUNT(*) AS rows_, COUNT(DISTINCT _file) AS files,
       COUNT_IF(value IS NULL) AS null_values, COUNT_IF(status IS NOT NULL) AS flagged
FROM laddstolpar_df.bronze.scb_tab628
UNION ALL
SELECT 'scb_tab3276', COUNT(*), COUNT(DISTINCT _file), COUNT_IF(value IS NULL), COUNT_IF(status IS NOT NULL)
FROM laddstolpar_df.bronze.scb_tab3276
UNION ALL
SELECT 'scb_tab3277', COUNT(*), COUNT(DISTINCT _file), COUNT_IF(value IS NULL), COUNT_IF(status IS NOT NULL)
FROM laddstolpar_df.bronze.scb_tab3277;

-- Heby: old code 1917 vs current 0331 in our period
SELECT region, COUNT(*) AS rows_, SUM(value) AS total, COUNT_IF(value IS NULL) AS nulls
FROM laddstolpar_df.bronze.scb_tab3277
WHERE region IN ('1917', '0331')
GROUP BY region;

In [0]:
%sql
-- Which owner category, year and flag?
SELECT agarkategori, tid, status, COUNT(*) AS n
FROM laddstolpar_df.bronze.scb_tab3276
WHERE value IS NULL
GROUP BY ALL
ORDER BY n DESC;

-- Which regions escape it (if 314 = 315 − 1)?
SELECT region, COUNT_IF(value IS NULL) AS nulls, COUNT(*) AS rows_
FROM laddstolpar_df.bronze.scb_tab3276
GROUP BY region
HAVING nulls = 0 OR nulls > 1
ORDER BY nulls
LIMIT 20;

In [0]:
lab = m3276["dimension"]["Agarkategori"]["category"]["label"]
print("Ägarkategori:", lab)

In [0]:
%sql
SELECT region, COUNT(*) AS nulls
FROM laddstolpar_df.bronze.scb_tab3276
WHERE value IS NULL AND agarkategori <> '040'
GROUP BY region;
-- expect exactly 15, 16, 1917 with 36 each

03 Trafikanalys – explore, län list vs SCB, bronze checks

In [0]:
TRAFA_NAME = "Trafikanalys"
TRAFA = "https://api.trafa.se/api/data"

def trafa(query):
    return http_call(TRAFA_NAME, f"GET query={query}", "GET", TRAFA, timeout=60, params={"query": query, "lang": "sv"}).json()

def trafa_rows(resp):
    """Rows -> dicts: dimensions keep code (Name) + label (Value); measures keep the raw Value."""
    out = []
    for row in resp.get("Rows", []):
        rec = {}
        for c in row["Cell"]:
            if c["IsMeasure"]:
                rec[c["Column"]] = c["Value"]
            else:
                rec[c["Column"]] = c["Name"]
                rec[c["Column"] + "_label"] = c["Value"]
        out.append(rec)
    return out

# A) one county, all its kommuner (no regkom values listed), 2 years, BEV + total
a = trafa("t10026|ar:2024,2025|reglan:01|regkom|drivmedel:103,t1|itrfslut")
print("Top-level keys:", list(a.keys()))
rows = a.get("Rows", [])
print("A) rows:", len(rows), "  (Stockholm county has 26 kommuner -> expect 26 × 2 × 2 = 104)")
print(json.dumps(rows[:2], ensure_ascii=False, indent=2)[:1500])

# B) same, but all fuel types (no drivmedel values listed)
b = trafa("t10026|ar:2025|reglan:01|regkom|drivmedel|itrfslut")
print("B) rows:", len(b.get("Rows", [])), "  (expect 26 × 9 = 234 if 'Totalt' is included)")

In [0]:
ra, rb = trafa_rows(a), trafa_rows(b)

print("A regkom (code, label): count")
for k, n in sorted(Counter((r["regkom"], r["regkom_label"]) for r in ra).items()):
    if n != 4: print("   ", k, n)          # only the odd ones; normal kommuner have 4
print("A distinct regkom:", len({r["regkom"] for r in ra}))

print("\nB drivmedel:", sorted(Counter((r["drivmedel"], r["drivmedel_label"]) for r in rb).items()))
print("B rows per regkom (odd ones):",
      {k: n for k, n in Counter((r["regkom"], r["regkom_label"]) for r in rb).items() if n != 9})
print("B empty measure values:", sum(1 for r in rb if r.get("itrfslut") in (None, "")))
print("Errors:", a.get("Errors"), "| Notes:", a.get("Notes"))

In [0]:
for code, label in sorted({(r["regkom"], r["regkom_label"]) for r in ra}):
    print(code, label)

In [0]:
# The constant LAN in 03_landing_trafa must equal the län in SCB TAB628 (this check is what lets 03 skip the SCB dependency)
LAN = ["01", "03", "04", "05", "06", "07", "08", "09", "10", "12", "13",
       "14", "17", "18", "19", "20", "21", "22", "23", "24", "25"]   # copy of the constant in 03_landing_trafa
scb_lan = sorted(r[0] for r in spark.sql("""
            SELECT DISTINCT region FROM laddstolpar_df.bronze.scb_tab628
            WHERE length(region) = 2 AND region <> '00'""").collect())
print("Only in constant:", sorted(set(LAN) - set(scb_lan)), "| only in SCB:", sorted(set(scb_lan) - set(LAN)))
assert LAN == scb_lan, "LAN constant in 03_landing_trafa is out of date vs bronze.scb_tab628"

In [0]:
%sql
SELECT COUNT(*)                                            AS rows_,
       COUNT(DISTINCT _file)                               AS files,
       COUNT(DISTINCT CASE WHEN regkom <> 't1' THEN regkom END) AS kommuner,
       COUNT_IF(regkom = 't1')                             AS county_total_rows,
       COUNT_IF(itrfslut IS NULL OR itrfslut = '')         AS empty_values,
       COUNT_IF(NOT itrfslut RLIKE '^[0-9]+$')             AS non_numeric
FROM laddstolpar_df.bronze.trafa_t10026;

-- Same kommuner in both sources? (expect 0 rows)
WITH t AS (SELECT DISTINCT regkom AS code FROM laddstolpar_df.bronze.trafa_t10026 WHERE regkom <> 't1'),
     s AS (SELECT DISTINCT region AS code FROM laddstolpar_df.bronze.scb_tab628 WHERE length(region) = 4)
SELECT 'only in Trafikanalys' AS side, code FROM t WHERE code NOT IN (SELECT code FROM s)
UNION ALL
SELECT 'only in SCB', code FROM s WHERE code NOT IN (SELECT code FROM t);

04 seeds – SKR workbook sheets, seed checks, BOM check

In [0]:
p = Path("/Volumes/laddstolpar_df/landing/raw/seeds/skr_kommungrupp/kommungruppsindelning-2023.xlsx")
xl = pd.ExcelFile(p)                                          # needs openpyxl: %pip install openpyxl
print("Sheets:", xl.sheet_names)
for s in xl.sheet_names:
    print(f"\n--- {s}")
    print(pd.read_excel(p, sheet_name=s, header=None, nrows=8).to_string())

In [0]:
%sql
-- 1) Lookup codes exist and names match SKR
SELECT o.kommun_kod, o.kommun_namn, s.kommun_namn AS skr_namn
FROM laddstolpar_df.bronze.seed_elomrade_kommun_override o
LEFT JOIN laddstolpar_df.bronze.seed_skr_kommungrupp s USING (kommun_kod)
WHERE s.kommun_kod IS NULL OR s.kommun_namn <> o.kommun_namn;

-- 2) SKR and SCB agree on the 290 kommuner
WITH scb AS (SELECT DISTINCT region AS kommun_kod FROM laddstolpar_df.bronze.scb_tab628 WHERE length(region) = 4)
SELECT 'only SKR' AS side, kommun_kod FROM laddstolpar_df.bronze.seed_skr_kommungrupp
WHERE kommun_kod NOT IN (SELECT kommun_kod FROM scb)
UNION ALL
SELECT 'only SCB', kommun_kod FROM scb
WHERE kommun_kod NOT IN (SELECT kommun_kod FROM laddstolpar_df.bronze.seed_skr_kommungrupp);

-- 3) Every kommun gets a zone
SELECT COALESCE(o.elomrade_primary, l.elomrade_default) AS elomrade, COUNT(*) AS kommuner
FROM laddstolpar_df.bronze.seed_skr_kommungrupp k
LEFT JOIN laddstolpar_df.bronze.seed_elomrade_kommun_override o USING (kommun_kod)
LEFT JOIN laddstolpar_df.bronze.seed_elomrade_lan_default l ON l.lan_kod = substr(k.kommun_kod, 1, 2)
GROUP BY 1
ORDER BY 1;

In [0]:
SEED_SRC = Path.cwd() / "seeds"                     # the Git folder; the notebook sits in the repo root
print((SEED_SRC / "param_scenario_weights.csv").read_bytes()[:30])   # BOM check (b'\xef\xbb\xbf' = BOM)

05 NOBIL – verification queries live in nobil_explore (last two cells).